In [1]:
import sys
import os
from dotenv import load_dotenv
from pathlib import Path
import pandas as pd

# if notebook is in PRIN/notebooks, parent() is PRIN
#project_root = Path.cwd().resolve().parent
#sys.path.insert(0, str(project_root))
#print("Added to sys.path:", project_root)

import json
from constants import Annotations, AnnotatedReport
import time
from IPython.display import clear_output

from huggingface_hub import login

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from transformers import AutoTokenizer, DefaultDataCollator
from datasets import load_dataset, Dataset, DatasetDict

from model_utils import create_label_to_id_map, from_output_to_labels

from train_utils import get_device, add_nan_flag_to_df, create_list_of_annotated_reports, create_hugging_face_dataset, add_target_columns_to_dataset, get_normalization_stats
from classifiers import ReportExtractor
from tqdm import tqdm
from sklearn.metrics import accuracy_score
import numpy as np

In [2]:
############
# Parameters
############
# Data parameters
TRAIN_FILE_NAME = "train_split.csv"
VALIDATION_FILE_NAME = "validation_split.csv"
TEST_FILE_NAME = "test_split.csv"

In [3]:
base_dir = Path.cwd().parent

In [4]:
# Set device
device = get_device()
print(f'{device = }')

torch.cuda.is_available() = False
torch.cuda.device_count() = 0
device = device(type='cpu')


In [5]:
model = ReportExtractor.from_pretrained(base_dir / "saved_models" / "report_extractor", device=device)
model.to(device)
model.eval()
tokenizer = AutoTokenizer.from_pretrained(base_dir / "saved_models" / "report_extractor")

c:\Users\lucat\PythonProjects\PRIN\ENCODER\classifiers.py:111: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(os.path.join(load_directory, "report_ext

In [6]:
###########
# Load data
###########
file_names = {
    'train': TRAIN_FILE_NAME,
    'validation': VALIDATION_FILE_NAME,
    'test': TEST_FILE_NAME
}
paths = {split: base_dir / "data" / file_name
         for split, file_name in file_names.items()}
data = {split: pd.read_csv(path) for split, path in paths.items()}
train_data, validation_data, test_data = data['train'], data['validation'], data['test']
# Log
print(f"{len(train_data) = }")
print(f"{len(validation_data) = }")
print(f"{len(test_data) = }")
# Create nan-flag columns
for split, df in data.items():
    add_nan_flag_to_df(df)
# Create lists of Annotated reports
annotated_reports =  {split: create_list_of_annotated_reports(data[split]) for split in file_names}

len(train_data) = 167
len(validation_data) = 46
len(test_data) = 56


In [7]:
# Check the maximum number of tokens for each report
print('model context length:', model.encoder.config.max_position_embeddings)
for split in annotated_reports:
    print(f'{split}: {len(annotated_reports[split])} reports')
    max_n_tokens = 0
    del_list = []
    for i, report in enumerate(annotated_reports[split]):
        x = tokenizer(report.report_text, return_tensors='pt')['input_ids'].shape[1]
        max_n_tokens = max(max_n_tokens, x)
        if x > model.encoder.config.max_position_embeddings:
            del_list.append(i)
    print(del_list)
    print(f'{max_n_tokens = }')
    # Delete long reports
    for i in del_list[::-1]:
        annotated_reports[split].pop(i)
    print(f'After deletion: {len(annotated_reports[split])} reports')

Token indices sequence length is longer than the specified maximum sequence length for this model (1427 > 512). Running this sequence through the model will result in indexing errors


model context length: 512
train: 167 reports
[0, 2, 10, 25, 30, 33, 38, 77, 91, 93, 97, 98, 99, 115, 116, 127, 130, 134, 148, 149, 150, 158, 161, 162, 163, 164, 166]
max_n_tokens = 1427
After deletion: 140 reports
validation: 46 reports
[7, 11, 23, 33, 34, 37, 38, 42, 44]
max_n_tokens = 1008
After deletion: 37 reports
test: 56 reports
[2, 6, 16, 23, 26, 28, 33, 37, 42, 43, 45, 47]
max_n_tokens = 861
After deletion: 44 reports


In [8]:
##############################
# Create Hugging Face Datasets
##############################
dataset = DatasetDict({
    split: create_hugging_face_dataset(annotated_reports[split])
    for split in annotated_reports
})
# Tokenize text and set format to torch
def tokenize_function(examples):
    return tokenizer(examples['text'], padding="longest", max_length=model.encoder.config.max_position_embeddings)
dataset = dataset.map(tokenize_function, batched=True)
cols_to_remove = [col for col in ["token_type_ids"] if col in dataset['validation'].column_names]
dataset = dataset.remove_columns(cols_to_remove)
dataset.set_format('torch')
# Log before adding columns
print(dataset)
# Add annotation fields to the dataset
label_to_id_map = create_label_to_id_map(model.annotations_model)
normalization_stats = get_normalization_stats(annotated_reports['train'], model.regression_fields)
for split, reports in annotated_reports.items():
    dataset[split] = add_target_columns_to_dataset(dataset=dataset[split],
                                                   annotated_reports=reports,
                                                   label_to_id_map=label_to_id_map,
                                                   classification_columns=model.classification_fields,
                                                   binary_classification_columns=model.binary_classification_fields,
                                                   multiple_choice_columns=model.multiple_choice_fields,
                                                   regression_columns=model.regression_fields,
                                                   normalization_stats=normalization_stats)
# Log after adding columns
print(dataset)

Map:   0%|          | 0/140 [00:00<?, ? examples/s]

Map:   0%|          | 0/37 [00:00<?, ? examples/s]

Map:   0%|          | 0/44 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'input_ids', 'attention_mask'],
        num_rows: 140
    })
    validation: Dataset({
        features: ['text', 'input_ids', 'attention_mask'],
        num_rows: 37
    })
    test: Dataset({
        features: ['text', 'input_ids', 'attention_mask'],
        num_rows: 44
    })
})
DatasetDict({
    train: Dataset({
        features: ['text', 'input_ids', 'attention_mask', 'morfologia', 'riflessione_peritoneale_anteriore', 'infiltrazione_tessuto_adiposo', 'infiltrazione_sfinteri', 'infiltrazione_organi_extra', 'coinvolgimento_riflessione_peritoneale', 'coinvolgimento_fascia_mesorettale', 'depositi_tumorali', 'emvi_esteso', 'stadio_T', 'stadio_N', 'mrf', 'emvi', 'metastasi', 'ore_inizio_is_nan', 'ore_fine_is_nan', 'spessore_parietale_is_nan', 'estensione_cranio_caudale_is_nan', 'distanza_oai_is_nan', 'numero_linfonodi_non_conosciuto', 'stadio_N1c', 'ore_inizio', 'ore_fine', 'spessore_parietale', 'estensione_cranio_caudale', 

In [9]:
input_ids = dataset['test'][:2]['input_ids'].to(device)
attention_mask = dataset['test'][:2]['attention_mask'].to(device)

In [10]:
output = model(input_ids=input_ids, attention_mask=attention_mask)

In [11]:
x = from_output_to_labels(model_output=output,
                      regression_fields=model.regression_fields,
                      binary_fields=model.binary_classification_fields,
                      classification_fields=model.classification_fields,
                      multiple_choice_fields=model.multiple_choice_fields,
                      label_to_id_map=label_to_id_map,
                      normalization_stats=normalization_stats)

In [12]:
x

{'numero_depositi': tensor([[0.0666],
         [0.0181]])}

In [43]:
output['linfonodi_sospetti'].shape

torch.Size([44, 1])

In [44]:
output['posizione'].shape

torch.Size([44, 4])

In [45]:
output['morfologia'].shape

torch.Size([44, 4])

In [ ]:
toutput['posizione'])

torch.Tensor

In [11]:
dataset['test'][:1]

{'text': ["RM ADDOME INFERIORE L'ESAME STATO ESEGUITO MEDIANTE SEQUENZE FSE,DWI, MIRATO ALLA STADIAZIONE LOCOREGIONALE DELLA NEOPLASIA RETTALE. L'ESAME STATO ESEGUITO IN DIFFICILI CONDIZIONI TECNICHE PER LA SCARSA COLLABORAZIONE DEL PAZIENTE E PER TALE MOTIVO IN ALCUNE SEQUENZE SONO PRESENTI ARTEFATTI DA MOVIMENTO. IN CORRISPONDENZA DEL RETTO MEDIO-BASSO PRESENTE GROSSOLANA FORMAZIONE AGGETTANTE CON EPICENTRO IN CORRISPONDENZA DELLA PARETE POSTEROLATERALE DESTRA CHE OCCUPA 3/4 DELLA CIRCONFERENZA DEL LUME CON ESTENSIONE CAUDO-CRANIALE DI CIRCA 6,8 CM DA CIRCA 4 CM DALLO SFINTERE ANALE INTERNO. LA NEOPLASIA INVIA DIGITAZIONI POLIPOIDI NEL MESORETTO DI DESTRA ED ANTERIORE CHE SEGUONO I VASI EMORROIDARI, ALCUNE DELLE QUALI ANTERIORMENTE RAGGIUNGONO LA FASCIA PERIRETTALE. TRE LINFONODI SOSPETTI NEL MESORETTO UNO DEI QUALI DI CIRCA 7 MM ADESO ALLA FASCIA PERIRETTALE. PICCOLO LINFONODO IN SEDE OTTURATORIA DESTRA. NUMEROSI, GROSSOLANI DIVERTICOLI CON AMPIO COLLETTO IN CORRISPONDENZA DEL SIGMA

In [12]:
annotated_reports['test'][0].report_data.model_dump()

{'morfologia': 'solido_polipoide',
 'posizione': ['medio', 'basso'],
 'ore_inizio': None,
 'ore_inizio_is_nan': True,
 'ore_fine': None,
 'ore_fine_is_nan': True,
 'spessore_parietale': None,
 'spessore_parietale_is_nan': True,
 'estensione_cranio_caudale': 66,
 'estensione_cranio_caudale_is_nan': False,
 'distanza_oai': 40.0,
 'distanza_oai_is_nan': False,
 'riflessione_peritoneale_anteriore': None,
 'infiltrazione_tessuto_adiposo': 'si_5mm_plus',
 'infiltrazione_sfinteri': None,
 'infiltrazione_organi_extra': None,
 'infiltrazione_organi_dettagli': [],
 'coinvolgimento_riflessione_peritoneale': None,
 'coinvolgimento_fascia_mesorettale': 'si',
 'linfonodi_sospetti': 4,
 'numero_linfonodi_non_conosciuto': False,
 'sedi_linfonodi': ['mesorettali', 'otturatori'],
 'depositi_tumorali': None,
 'numero_depositi': 0,
 'emvi_esteso': 'si',
 'stadio_T': 'T3cd',
 'stadio_N': 'N2a',
 'stadio_N1c': False,
 'mrf': '+',
 'emvi': '+',
 'metastasi': None}